In [1]:
# SAML-D 프로파일 — 스키마, 건수, 기간, 값 종류, 라벨 분포
import duckdb, pandas as pd
from IPython.display import display

CSV = "/workspace/money_laundry_ai_/datasets/SAML-D/SAML-D.csv"   # 받은 위치로 바꾸기
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE VIEW s AS SELECT * FROM read_csv_auto('{CSV}', header=true)")
display(con.execute("DESCRIBE s").df())                                          # 컬럼 이름·타입

display(con.execute("""
    SELECT count(*) AS 거래, sum(Is_laundering) AS 세탁, 100 * avg(Is_laundering) AS 세탁_비율_pct,
           min(Date) AS 시작일, max(Date) AS 종료일, count(DISTINCT Date) AS 날짜_수,
           count(DISTINCT Sender_account) AS 송신_계좌, count(DISTINCT Receiver_account) AS 수신_계좌,
           min(Amount) AS 최소_금액, median(Amount) AS 중앙_금액, max(Amount) AS 최대_금액
    FROM s
""").df().T)

for c in ["Payment_type", "Payment_currency", "Received_currency", "Sender_bank_location", "Receiver_bank_location"]:
    print(f"── {c}")
    display(con.execute(f"""SELECT {c}, count(*) AS 거래, sum(Is_laundering) AS 세탁,
                                   round(100 * avg(Is_laundering), 4) AS 세탁_비율_pct
                            FROM s GROUP BY 1 ORDER BY 2 DESC""").df())

print("── Laundering_type (정상·의심 유형별)")
display(con.execute("""SELECT Is_laundering, Laundering_type, count(*) AS 거래
                       FROM s GROUP BY ALL ORDER BY Is_laundering DESC, 거래 DESC""").df())

print("── 월별 거래·세탁")
display(con.execute("""SELECT strftime(CAST(Date AS DATE), '%Y-%m') AS 월, count(*) AS 거래, sum(Is_laundering) AS 세탁
                       FROM s GROUP BY 1 ORDER BY 1""").df())
display(con.execute("SELECT * FROM s LIMIT 5").df())

IOException: IO Error: No files found that match the pattern "/workspace/money_laundry_ai_/datasets/SAML-D/SAML-D.csv"

LINE 1: CREATE OR REPLACE VIEW s AS SELECT * FROM read_csv_auto('/workspace/money_laundry_ai_/datasets/SAML...
                                                  ^